## Benchmark features of `future` submodule .vs. legacy code.

In [ ]:
import numpy as np

from yeti_iga.future.bspline import (BSpline, BSplineSurface, ControlPointManager,
    Patch, SubdivisionRefiner, PRefiner, GlobalDOFManager, PatchDOFManager,
    IGABasis1D, PatchIntegrator, MaterialProperties)

from yeti_iga.preprocessing.igaparametrization import IGAparametrization
from yeti_iga.stiffmtrx_elemstorage import sys_linmat_lindef_static as build_stiffmatrix
from yeti_iga.stiffmtrx_elemstorage_omp import sys_linmat_lindef_static_omp as build_stiffmatrix_omp


In [ ]:
# Create data for future submodule
mgr = ControlPointManager(dim=2)
mgr.add_point([0.0, 1.0])
mgr.add_point([1.0, 1.0])
mgr.add_point([1.0, 0.0])
mgr.add_point([0.0, 2.0])
mgr.add_point([2.0, 2.0])
mgr.add_point([2.0, 0.0])

dofs_per_cp = [2 for _ in range(mgr.n_points)]
dof_manager = GlobalDOFManager(dofs_per_cp)

surf = BSplineSurface(
    BSpline(2, np.array([0., 0., 0., 1., 1., 1.])),
    BSpline(1, np.array([0., 0., 1., 1.]))
)

mapping = [i for i in range(6)]
dof_manager_patch = PatchDOFManager(2, mapping, dof_manager)
patch = Patch(surf, mgr, mapping, [3, 2], dof_manager_patch)



In [ ]:
# Create data for legacy yeti
iga_model = IGAparametrization(filename="input/QuarterDiskBS/QuarterDiskBS")

In [ ]:
# Build stiffness matrix for future submodule
basis_u = IGABasis1D.build(patch.tensor.components[0], 3)
basis_v = IGABasis1D.build(patch.tensor.components[1], 2)

integrator = PatchIntegrator(patch, basis_u, basis_v, MaterialProperties(210000., 0.3))

stiffness_matrix = integrator.integrate_stiffness()

In [ ]:
## Benchmark refinement
import datetime
import time

ref = np.array([3, 3])
deg = np.array([4, 5])

t0 = datetime.datetime.now()

# future (1D-first fast path: compose T_1d in 1D, apply CPs once at end)
_ = PRefiner(direction=0, n_elevations=deg[0]).refine_1d(patch)
_ = PRefiner(direction=1, n_elevations=deg[1]).refine_1d(patch)
_ = SubdivisionRefiner(direction=0, n_levels=ref[0]).refine_1d(patch)
_ = SubdivisionRefiner(direction=1, n_levels=ref[1]).refine_1d(patch)

t1 = datetime.datetime.now()

# legacy
iga_model.refine(nb_refinementByDirection=ref, nb_degreeElevationByDirection=deg)

t2 = datetime.datetime.now()

print('future:', t1 - t0, 'seconds')
print('legacy:', t2 - t1, 'seconds')

In [ ]:
print(mgr.n_points)

In [ ]:
# NB: `dof_manager` (GlobalDOFManager) was sized for the 6 initial control
# points and is NOT grown by refinement -- use patch.dof_manager
# (PatchDOFManager), which IS updated at every refine_1d() call.
for i in range(patch.n_cp):
    print(patch.dof_manager.get_global_dof_indices(i))